# Occultation-detection trade study

Interactive companion to `ColibriPipeline/detection_trade_study/`.

Inject synthetic **Fresnel** occultation signals (`fresnel_physics.genCurve`)
into **real** light curves (`stars.npy`, 2025-08-30 01.54.12) and compare
detectors, telescope-combination policies, and preprocessing methods.

Run `python -m detection_trade_study.run_trade_study` first to produce the
`results/` CSVs, or run the cells below to generate trials live.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'ColibriPipeline'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from detection_trade_study import harness, detectors as det_mod, injection as inj, bootstrap as bs

## Inspect the data and an injected signal

In [ ]:
data_by_scope, _, all_scopes = harness.load_all()
for s in all_scopes:
    print(s, 'flux shape', data_by_scope[s]['flux'].shape)
print('Green==Blue identical:', harness.scopes_identical(data_by_scope, ['Green','Blue']))
src = data_by_scope['Green']
hosts = harness.photometric_hosts(src['flux'], 500)
print(f'{len(hosts)} positive-flux host stars; median per-frame SNR',
      round(np.median([harness.lc.star_snr(src['flux'][i]) for i in hosts]),2))

prof = inj.make_profile({'startLam':4e-7,'endLam':7e-7,'objectRad':2000.,
                         'impact':0.,'dist':40.,'angDi':0.02,'shiftAdj':0.0})
f0 = src['flux'][hosts[0]].astype(float); f1 = inj.inject(f0, 1200, prof)
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].plot(f1); ax[0].set_title(f'host + injected dip (depth={inj.injected_depth(prof):.2f})')
ax[1].plot(range(1180,1240), f1[1180:1240], '.-'); ax[1].set_title('zoom on event')
plt.show()

## Detector comparison + power of three (bootstrapped independent telescopes)

Green/Blue are identical copies and Red is a 5 s partial, so the multi-telescope
test synthesises independent noise by block-bootstrapping one real star.

In [ ]:
detectors = det_mod.ALL_DETECTORS()
df2, sc2, ref2 = harness.run_trials_bootstrap(src, hosts, detectors, n_scopes=3,
                                              n_inject=150, n_null=150,
                                              rng=np.random.default_rng(0))
roc2 = harness.compute_roc(df2, sc2, ref2, detectors=list(detectors))
print(f'Power of {len(sc2)}: completeness at ~1% FAR')
print(f'{"detector":30s} {"1-scope":>8s} {"AND":>8s} {"joint":>8s}')
for det in detectors:
    s=harness.completeness_at_far(roc2[det]['single'],0.01)
    a=harness.completeness_at_far(roc2[det]['and'],0.01)
    j=harness.completeness_at_far(roc2[det]['joint'],0.01)
    print(f'{det:30s} {s:8.2f} {a:8.2f} {j:8.2f}')

## Preprocessing trade study + SNR-aware sensitivity

Why pooled completeness looked low: the noise is mostly high-frequency (the
boxcar detrend barely changes the std) and the host pool is dominated by faint
stars (median per-frame SNR ~4.5). So we trade-study preprocessing x shape and
report completeness binned by stellar SNR and event depth.

In [ ]:
from detection_trade_study.preprocessing import ALL_PREPROCESSORS
grid = det_mod.build_grid(fresnel_stride=80)
shapes = list(det_mod._shape_templates().keys()); preps = list(ALL_PREPROCESSORS())
gdf, gsc, gref = harness.run_trials_bootstrap(src, hosts, grid, n_scopes=1,
                                              n_inject=150, n_null=150,
                                              rng=np.random.default_rng(1))
groc = harness.compute_roc(gdf, gsc, gref, detectors=list(grid))
tab = pd.DataFrame({p:[harness.completeness_at_far(groc[f'{s}@{p}']['single'],0.01)
                       for s in shapes] for p in preps}, index=shapes)
print('Completeness @1% FAR (rows=shape, cols=preprocessing):'); print(tab.round(2))

In [ ]:
shape='RickerDetector'
fig, ax = plt.subplots(figsize=(7,5))
for p in preps:
    e = harness.completeness_vs_eventSNR(gdf, f'{shape}@{p}', gref, far_target=0.01)
    ax.plot(e['centers'], e['completeness'], marker='o', label=p)
ax.set_xscale('log'); ax.set_xlabel('event SNR ~ depth*SNR*sqrt(duration)')
ax.set_ylabel('completeness @1% FAR'); ax.set_title(f'{shape}: by preprocessing')
ax.legend(); ax.grid(alpha=.3); plt.show()

best = max(grid, key=lambda k: harness.completeness_at_far(groc[k]['single'],0.01))
cm = harness.completeness_map(gdf, best, gref)
plt.figure(figsize=(6,5))
plt.imshow(cm['completeness'], origin='lower', aspect='auto', vmin=0, vmax=1,
           extent=[cm['depth_edges'][0],cm['depth_edges'][-1],cm['snr_edges'][0],cm['snr_edges'][-1]])
plt.colorbar(label='completeness'); plt.xlabel('event depth'); plt.ylabel('stellar SNR')
plt.title(f'best combo: {best}'); plt.show()

### Extending the study

* New shape: add a dip-template factory in `detectors.py` + an entry in `_shape_templates()`.
* New preprocessing: subclass `preprocessing.Preprocessor`, return `Conditioned(series, noise)`, add to `ALL_PREPROCESSORS()`.
* Real 3-telescope data: call `harness.run_trials_real` with all three scopes (the joint then uses real, uncorrelated noise instead of the bootstrap).